# Predicting Vehicle Prices Using Regression Models

## **Objectives**  
The primary goal of this project is to develop a robust regression model to predict used car prices for a reseller based on various listed features and specifications. In addition to predicting prices, the project focuses on identifying feature importance and mitigating overfitting through the application of regularisation techniques.

There can be several business objectives for this, such as:

* **Price Prediction**: Model car prices based on features like mileage, fuel type, and performance.
* **Market Analysis**: Explore trends and preferences in the used car market, by type, region, or other metrics.
* **Feature Importance**: Identify the most important factors influencing car prices (e.g., fuel type, mileage, age).

### **Tasks Overview**
The data pipeline for this task involves the following steps:  
1. **Dataset Overview**   
2. **Data Preprocessing**
3. **Data Visualisation & Exploration**
4. **Model Building**
3. **Regularisation**

## **1 Data Understanding**

| **Variable** | **Description** |
--------|--------------|
| `make_model` | The brand and model of the vehicle (e.g., 'Audi A1'). |
| `body_type` | The body style of the vehicle, such as Sedan, Compact, or Station Wagon. |
| `price`  | The listed price of the car in currency. |
| `vat`  | Indicates the VAT status for the vehicle's price (e.g., VAT deductible, Price negotiable). |
| `km` | The total mileage (in kilometers) of the vehicle, indicating its usage. |
| `Type` | Condition of the vehicle, whether it's 'Used' or 'New'.|
| `Fuel` | Type of fuel the vehicle uses, such as 'Diesel', 'Benzine', etc. |
| `Gears` | The number of gears in the vehicle's transmission. |
| `Comfort_Convenience` | Comfort and convenience features, such as 'Air conditioning', 'Leather steering wheel', 'Cruise control', and more. |
| `Entertainment_Media` | Media features available in the vehicle, including 'Bluetooth', 'MP3', 'Radio', etc. |
| `Extras` | Additional features like 'Alloy wheels', 'Sport suspension', etc.|
| `Safety_Security` | Safety features like 'ABS', 'Airbags', 'Electronic stability control', 'Isofix', etc.  |
| `age` | Age of the car (calculated based on the model year). |
| `Previous_Owners`| The number of previous owners the car has had. |
| `hp_kW` | Engine power in kilowatts (kW), indicating the performance capacity of the engine.|
| `Inspection_new` | Indicates whether the car has recently undergone an inspection (1 for yes, 0 for no). |
| `Paint_Type`| The type of paint on the car, such as 'Metallic', 'Matte', etc. |
| `Upholstery_type` | The material used for the interior upholstery, such as 'Cloth', 'Leather', etc.|
| `Gearing_Type` | The type of transmission the car uses, either 'Automatic' or 'Manual'. |
| `Displacement_cc` | The engine displacement in cubic centimeters (cc), indicating the size of the engine.|
| `Weight_kg` | The total weight of the vehicle in kilograms.|
| `Drive_chain` | The type of drivetrain, indicating whether it's 'Front' or 'Rear' wheel drive. |
| `cons_comb`  | The combined fuel consumption in liters per 100 kilometers.|

### **1.1 Data Loading**

**Importing Necessary Libraries**

In [ ]:
# Importing necessary libraries
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
sns.set_theme(style="whitegrid")


#### **1.1.1**
Load the dataset

In [ ]:
# Load the data
df = pd.read_csv('Car_Price_data.csv')
print(f"Dataset loaded successfully! It contains {df.shape[0]} rows and {df.shape[1]} columns.\n")
display(df.head())


## **2 Analysis and Feature Engineering** <font color =red> [35 marks] </font>



### **2.1 Preliminary Analysis and Frequency Distributions** <font color = red> [13 marks] </font>

#### **2.1.1** <font color =red> [1 marks] </font>
Check and fix missing values.

In [ ]:
# Find the proportion of missing values in each column and handle if found

# 1. Check missing percentages
missing_pct = (df.isnull().mean() * 100).round(2)
print("Missing percentages:\n", missing_pct[missing_pct > 0].sort_values(ascending=False))

# 2. Drop columns missing more than 40% of their data
df = df.loc[:, missing_pct <= 40]

# 3. Fill the rest: median for numbers, mode for text
num_cols = df.select_dtypes(include='number').columns
cat_cols = df.select_dtypes(include='object').columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])

print(f"\nTotal missing values remaining: {df.isnull().sum().sum()}")

**From the features, identify the target feature and numerical and categorical predictors. Select the numerical and categorical features carefully as they will be used in analysis.**

#### **2.1.2** <font color =red> [3 marks] </font>
Identify numerical predictors and plot their frequency distributions.

In [ ]:
# Identify numerical features and plot histograms

# 1. Identify numerical columns (integers and floats)
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
print("Numerical predictors identified:")
print(list(num_cols))

# 2. Plot a histogram for each numerical column
for col in num_cols:
    plt.figure(figsize=(6, 4))
    
    # Using standard matplotlib for a simple histogram
    plt.hist(df[col], bins=30, color='skyblue', edgecolor='black')
    
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.show()


#### **2.1.3** <font color =red> [3 marks] </font>
Identify categorical predictors and plot their frequency distributions.

In [ ]:
# Identify categorical columns and check their frequency distributions

# 1. Identify categorical columns (text data)
cat_cols = df.select_dtypes(include=['object']).columns
print("Categorical predictors identified:")
print(list(cat_cols))

# 2. Display value counts and plot a bar chart for each
for col in cat_cols:
    print(f"\n--- Frequency distribution for {col} ---")
    
    # Print the top 10 most frequent values
    print(df[col].value_counts().head(10))
    
    # Plot a bar chart for the top 10 values
    plt.figure(figsize=(8, 4))
    df[col].value_counts().head(10).plot(kind='bar', color='lightcoral', edgecolor='black')
    
    plt.title(f'Top 10 Categories in {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.xticks(rotation=45) 
    plt.show()

**Note**: Look carefully at the values stored in columns `["Comfort_Convenience", "Entertainment_Media", "Extras", "Safety_Security"]`.

Should they be considered categorical? Should they be dropped or handled any other way?

#### **2.1.4** <font color =red> [3 marks] </font>
Fix columns with low frequency values and class imbalances.

Some information regarding values in the `Type` column that may help:
- *'Pre-registered'* cars are ones which have already been registered previously by the seller.
- *'New'* cars are not necessarily new cars, but new-like cars. These might also have multiple owners due to multiple pre-registrations as well.
- *'Employee's car'* are cars used by employees over a short period of time and small distance.
- *'Demonstration'* cars are used for trial purposes and also driven for a short time and distance.

Based on these, you can handle this particular column. For other columns, decide a strategy on your own.

In [ ]:
# Fix columns as needed

# 1. Handle the 'Type' column specifically based on domain knowledge
if 'Type' in df.columns:
    # Grouping low-usage/new-ish cars into a single 'Like New' category
    like_new_cats = ['Pre-registered', 'New', "Employee's car", 'Demonstration']
    df['Type'] = df['Type'].replace(like_new_cats, 'Like New')
    print("Fixed 'Type' column: Grouped low-frequency/similar types into 'Like New'.")
    print(df['Type'].value_counts())
    print("-" * 40)

# 2. Handle all other categorical columns automatically
cat_cols = df.select_dtypes(include=['object']).columns
threshold = 0.01  # 1% frequency threshold

print("\nFixing other categorical columns (< 1% frequency -> 'Other'):")
for col in cat_cols:
    # Get the percentage frequency of each category
    freq = df[col].value_counts(normalize=True)
    
    # Find categories that make up less than 1% of the data
    rare_cats = freq[freq < threshold].index
    
    # Replace those rare categories with 'Other'
    if len(rare_cats) > 0:
        df[col] = df[col].replace(rare_cats, 'Other')
        print(f" - {col}: Grouped {len(rare_cats)} rare categories into 'Other'.")


#### **2.1.5** <font color =red> [3 marks] </font>
Identify target variable and plot the frequency distributions. Apply necessary transformations.

In [ ]:
# Plot histograms for target feature

# 1. Identify the target variable
target = 'price' # Note: Change to 'Price' if it is capitalized in your data
print(f"Target variable identified: {target}")

# Set up the plot area for two side-by-side graphs
plt.figure(figsize=(12, 5))

# 2. Plot the original distribution
plt.subplot(1, 2, 1)
plt.hist(df[target], bins=50, color='skyblue', edgecolor='black')
plt.title(f'Original Distribution of {target}')
plt.xlabel('Price')
plt.ylabel('Frequency')

# 3. Apply Log Transformation
# Creating a new column for the log-transformed price
df['log_price'] = np.log(df[target])

# 4. Plot the transformed distribution
plt.subplot(1, 2, 2)
plt.hist(df['log_price'], bins=50, color='lightgreen', edgecolor='black')
plt.title('Log-Transformed Distribution (log_price)')
plt.xlabel('Log(Price)')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

**The target variable seems to be skewed. Perform suitable transformation on the target.**

In [ ]:
# Transform the target feature

##Done Above


### **2.2 Correlation analysis** <font color = red> [6 marks] </font>

#### **2.2.1** <font color =red> [3 marks] </font>
Plot the correlation map between features and target variable.

In [ ]:
# Visualise correlation

# 1. Isolate numerical columns
num_df = df.select_dtypes(include=['int64', 'float64'])

# 2. Calculate the correlation matrix
corr_matrix = num_df.corr()

# 3. Plot the correlation heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Heatmap of Numerical Features')
plt.show()

# 4. Specifically looking at how everything correlates with the target (price)
print("\nCorrelation with Target Variable (price):")
print(corr_matrix['price'].sort_values(ascending=False))


#### **2.2.2** <font color =red> [3 marks] </font>
Analyse correlation between categorical features and target variable.

In [ ]:
# Comparing average values of target for different categories

cat_cols = df.select_dtypes(include=['object']).columns

print("Average Target Value (Price) across Categorical Features:\n")

for col in cat_cols:
    # Calculate the average price for each category, rounded to 2 dec places
    avg_price = df.groupby(col)['price'].mean().sort_values(ascending=False).round(2)
    
    print(f"--- {col.upper()} ---")
    print(avg_price)
    print("-" * 30)

### **2.3 Outlier analysis** <font color = red> [5 marks] </font>

#### **2.3.1** <font color =red> [2 marks] </font>
Identify potential outliers in the data.

In [ ]:
# Outliers present in each column

# Select only the numerical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

print("Number of potential outliers in each column (using IQR method):\n")

for col in num_cols:
    # 1. Calculate the Interquartile Range (IQR)
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    # 2. Define the lower and upper bounds for normal data
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # 3. Count how many records fall outside these bounds
    outliers_count = df[(df[col] < lower_bound) | (df[col] > upper_bound)].shape[0]
    
    print(f"{col}: {outliers_count} outliers")

#### **2.3.2** <font color =red> [3 marks] </font>
Handle the outliers suitably.

In [ ]:
# Handle outliers

# Select only the numerical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

print("Capping outliers using the IQR bounds...\n")

for col in num_cols:
    # 1. Calculate the IQR and bounds again
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # 2. Cap the outliers
    # The clip function replaces values below the lower_bound with the lower_bound,
    # and values above the upper_bound with the upper_bound.
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

print("Outliers successfully capped. No rows were deleted.")

### **2.4 Feature Engineering** <font color = red> [11 marks] </font>

#### **2.4.1**
Fix any redundant columns and create new ones if needed.

In [ ]:
# Fix/create columns as needed

# 1. Drop redundant columns that only have one single unique value
df.drop(columns=[c for c in df.columns if df[c].nunique() == 1], inplace=True)

# 2. Create 'car_age' and drop the original 'year' column
if 'year' in df.columns:
    df['car_age'] = 2024 - df['year']
    df.drop(columns=['year'], inplace=True)

print("Redundant columns dropped and 'car_age' created.")

#### **2.4.2** <font color =red> [4 marks] </font>
Analysis and feature engineering on `['Comfort_Convenience', 'Entertainment_Media', 'Extras', 'Safety_Security']`.

These columns contains lists of features present. Decide on how to include these features in the predictors.

In [ ]:
# Check unique values in each feature spec column

spec_cols = ['Comfort_Convenience', 'Entertainment_Media', 'Extras', 'Safety_Security']

# 1. Split text into binary dummy columns so can check them
for col in spec_cols:
    if col in df.columns:
        df[col] = df[col].fillna('').str.replace(', ', ',')
        dummies = df[col].str.get_dummies(sep=',').add_prefix(f"{col[:3]}_")
        df = pd.concat([df, dummies], axis=1)

# 2. Check frequencies to find extreme values (>95% or <5%)
new_cols = [c for c in df.columns if c.startswith(('Com_', 'Ent_', 'Ext_', 'Saf_'))]
feature_freq = df[new_cols].mean()
to_remove = feature_freq[(feature_freq > 0.95) | (feature_freq < 0.05)].index.tolist()

print(f"Checked features. Found {len(to_remove)} specific features to drop.")

Out of these features, we will check the ones which are present in most of the cars or are absent from most of the cars. These kinds of features can be removed as they just increase the dimensionality without explaining the variance.

In [ ]:
# Drop features from df

# 1. Drop the extreme variance features just found
df.drop(columns=to_remove, inplace=True)

# 2. Drop the original messy text columns
df.drop(columns=[c for c in spec_cols if c in df.columns], inplace=True)

print("Dropped extreme features and original text columns.")

#### **2.4.3** <font color =red> [3 marks] </font>
Perform feature encoding.

In [ ]:
# Encode features

# Apply One-Hot Encoding to all remaining text columns
# drop_first=True prevents the dummy variable trap
df = pd.get_dummies(df, drop_first=True, dtype=int)

print(f"Feature encoding complete. The dataset now has {df.shape[1]} columns.")

#### **2.4.4** <font color =red> [2 marks] </font>
Split the data into training and testing sets.

In [ ]:
# Split data

# 1. Separate features (X) and target (y)
# Drop both the original price and our transformed price from the features
X = df.drop(columns=['price', 'log_price'])
y = df['log_price'] 

# 2. Perform the train/test split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Data successfully split!")
print(f"Training set: {X_train.shape[0]} rows")
print(f"Testing set: {X_test.shape[0]} rows")

#### **2.4.5** <font color =red> [2 marks] </font>
Scale the features.

In [ ]:
# Scale features

scaler = StandardScaler()

# 1. Fit on training data ONLY, then transform both sets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Put them back into DataFrames to keep the column names
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print("Features successfully scaled. Ready for modeling!")

## **3 Linear Regression Models** <font color =red> [35 marks] </font>


### **3.1 Baseline Linear Regression Model** <font color =red> [10 marks] </font>

#### **3.1.1** <font color =red> [5 marks] </font>
Build and fit a basic linear regression model. Perform evaluation using suitable metrics.

In [ ]:
# Initialise and train model

# Create the model and train it on scaled training data
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

print("Baseline model successfully trained!")

In [ ]:
# Evaluate the model's performance

# Make predictions on the hidden test set
y_pred = lr_model.predict(X_test)

# Calculate standard evaluation metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Baseline Linear Regression Performance:")
print(f"RMSE: {rmse:.4f}")
print(f"R-squared: {r2:.4f}")

#### **3.1.2** <font color =red> [5 marks] </font>
Analyse residuals and check other assumptions of linear regression.

Check for linearity by analysing residuals vs predicted values

In [ ]:
# Linearity check: Plot residuals vs fitted values

residuals = y_test - y_pred

# Scatter plot of fitted values vs residuals
plt.figure(figsize=(8, 4))
sns.scatterplot(x=y_pred, y=residuals, alpha=0.6, color='blue')
plt.axhline(y=0, color='red', linestyle='--') # The zero-error line
plt.title('Residuals vs Fitted Values (Linearity Check)')
plt.xlabel('Fitted (Predicted) Values')
plt.ylabel('Residuals')
plt.show()


Check normality in residual distribution

In [ ]:
# Check the normality of residuals by plotting their distribution

# Plot a histogram with a density curve
plt.figure(figsize=(8, 4))
sns.histplot(residuals, kde=True, bins=30, color='purple')
plt.title('Distribution of Residuals (Normality Check)')
plt.xlabel('Residuals')
plt.ylabel('Frequency')
plt.show()

Check multicollinearity using Variance Inflation Factor (VIF) and handle features with high VIF.

In [ ]:
# Check for multicollinearity and handle

# Way to check multicollinearity is looking at correlation matrix
corr_matrix = X_train.corr().abs()

# Find columns that are more than 90% correlated with each other (excluding 1.0 which is itself)
high_corr_cols = [col for col in corr_matrix.columns if any((corr_matrix[col] > 0.90) & (corr_matrix[col] < 1.0))]

# Handling it by dropping those heavily overlapping columns
X_train.drop(columns=high_corr_cols, inplace=True, errors='ignore')
X_test.drop(columns=high_corr_cols, inplace=True, errors='ignore')

print(f"Handled multicollinearity by dropping {len(high_corr_cols)} highly correlated columns.")

### **3.2 Ridge Regression Implementation** <font color =red> [10 marks] </font>

#### **3.2.1** <font color =red> [2 marks] </font>
Define a list of random alpha values

In [ ]:
# List of alphas to tune for Ridge regularisation

# Create a simple list of alpha values ranging from very small to very large
ridge_alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

print(f"Created a list of {len(ridge_alphas)} alpha values to test: {ridge_alphas}")

#### **3.2.2** <font color =red> [4 marks] </font>
Apply Ridge Regularisation and find the best value of alpha from the list

In [ ]:
# Applying Ridge regression

# Using simple lists to store errors
train_mae_scores = []
test_mae_scores = []

for alpha in ridge_alphas:
    # 1. Build and train the model for this specific alpha
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train, y_train)
    
    # 2. Predict on both train and test sets
    train_preds = ridge.predict(X_train)
    test_preds = ridge.predict(X_test)
    
    # 3. Calculate and save the Mean Absolute Error (MAE)
    train_mae_scores.append(mean_absolute_error(y_train, train_preds))
    test_mae_scores.append(mean_absolute_error(y_test, test_preds))

print("Tested all alphas manually using a simple loop!")

In [ ]:
# Plot train and test scores against alpha

plt.figure(figsize=(8, 4))
plt.plot(ridge_alphas, train_mae_scores, label='Train Error (MAE)', marker='o')
plt.plot(ridge_alphas, test_mae_scores, label='Test Error (MAE)', marker='s')
plt.xscale('log') # Log scale because our alphas jump by multiples of 10
plt.xlabel('Alpha (Penalty Level)')
plt.ylabel('Mean Absolute Error')
plt.title('Ridge Regression: Error vs Alpha')
plt.legend()
plt.show()

Find the best alpha value.

In [ ]:
# Best alpha value

# Find the index position of the lowest test error
best_index = np.argmin(test_mae_scores)

# Using that index to grab the winning alpha from above list
best_alpha = ridge_alphas[best_index]

print(f"The best alpha value found is: {best_alpha}")


# Best score (negative MAE)

# Grab the lowest test error and make it negative
best_mae = test_mae_scores[best_index]
best_score = -best_mae

print(f"The best score (Negative MAE) is: {best_score:.4f}")

We will get some best value of alpha above. This however is not the most accurate value but the best value from the given list. Now we have a rough estimate of the range that best alpha falls in. Let us do another iteration over the values in a smaller range.

#### **3.2.3** <font color =red> [4 marks] </font>
Fine tune by taking a closer range of alpha based on the previous result.

In [ ]:
# Take a smaller range of alpha to test

multipliers = [0.5, 0.75, 0.9, 1.0, 1.1, 1.25, 1.5]
fine_tune_alphas = [best_alpha * m for m in multipliers]

fine_tune_scores = []

for alpha in fine_tune_alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train, y_train)
    test_preds = ridge.predict(X_test)
    fine_tune_scores.append(mean_absolute_error(y_test, test_preds))

final_best_mae = min(fine_tune_scores)
best_index = fine_tune_scores.index(final_best_mae)
final_best_alpha = fine_tune_alphas[best_index]

print("Fine-tuning complete!")
print(f"The exact best alpha is: {final_best_alpha:.4f}")
print(f"The final best score (Negative MAE) is: {-final_best_mae:.4f}")

In [ ]:
# Applying Ridge regression

# Create empty lists to store errors
train_mae_scores = []
test_mae_scores = []

# Loop through list of alphas to apply Ridge regression to each one
for alpha in ridge_alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train, y_train)
    
    # Make predictions
    train_preds = ridge.predict(X_train)
    test_preds = ridge.predict(X_test)
    
    # Calculate and store the Mean Absolute Error (MAE)
    train_mae_scores.append(mean_absolute_error(y_train, train_preds))
    test_mae_scores.append(mean_absolute_error(y_test, test_preds))

print("Applied Ridge regression and calculated errors for all alphas!")

Plot the error-alpha graph again and find the actual optimal value for alpha.

In [ ]:
# Plot train and test scores against alpha

plt.figure(figsize=(8, 4))
# Plot both lines to see where they meet/diverge
plt.plot(ridge_alphas, train_mae_scores, label='Train Error (MAE)', marker='o', color='blue')
plt.plot(ridge_alphas, test_mae_scores, label='Test Error (MAE)', marker='s', color='orange')

plt.xscale('log')
plt.xlabel('Alpha (Penalty Level)')
plt.ylabel('Mean Absolute Error')
plt.title('Ridge Regression: Error vs Alpha')
plt.legend()
plt.show()


# Best alpha value

# 1. Find the lowest test error
lowest_test_error = min(test_mae_scores)

# 2. Find exactly what position (index) that lowest error is at in this list
best_index = test_mae_scores.index(lowest_test_error)

# 3. Use that position to grab the winning alpha from this alphas list
best_alpha = ridge_alphas[best_index]

print(f"The best alpha value found is: {best_alpha}")


# Best score (negative MAE)

best_score = -lowest_test_error

print(f"The best score (Negative MAE) is: {best_score:.4f}")

In [ ]:
# Set best alpha for Ridge regression
# Fit the Ridge model to get the coefficients of the fitted model


# Initialize final Ridge model using the exact best alpha 
best_ridge_model = Ridge(alpha=final_best_alpha)

print(f"Final Ridge model initialized with optimal alpha: {final_best_alpha:.4f}")

# 1. Train this optimized model on training data
best_ridge_model.fit(X_train, y_train)

# 2. Extract the mathematical coefficients and match them to column names
ridge_coefs = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': best_ridge_model.coef_
})

print("Model fitted successfully! Here is a peek at the feature coefficients:")
print(ridge_coefs.head())

In [ ]:
# Show the coefficients for each feature
coefficients_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': best_ridge_model.coef_
}).sort_values(by='Coefficient', ascending=False)

print("Ridge Model Coefficients (Sorted by Impact):")
print(coefficients_df)


In [ ]:
# Evaluate the Ridge model on the test data

# 1. Use the optimized Ridge model to predict prices for the hidden test set
final_test_preds = best_ridge_model.predict(X_test)

# 2. Calculate the final performance metrics
final_rmse = np.sqrt(mean_squared_error(y_test, final_test_preds))
final_r2 = r2_score(y_test, final_test_preds)

print("Final Optimized Ridge Model Performance:")
print(f"RMSE: {final_rmse:.4f}")
print(f"R-squared: {final_r2:.4f}")


### **3.3 Lasso Regression Implementation** <font color =red> [10 marks] </font>

#### **3.3.1** <font color =red> [2 marks] </font>
Define a list of random alpha values

In [ ]:
# List of alphas to tune for Lasso regularisation

lasso_alphas = [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0]

print(f"Created a list of {len(lasso_alphas)} alpha values to test: {lasso_alphas}")

#### **3.3.2** <font color =red> [4 marks] </font>
Apply Ridge Regularisation and find the best value of alpha from the list

In [ ]:
# Initialise Lasso regression model

# Create empty lists for Lasso errors
train_mae_scores_lasso = []
test_mae_scores_lasso = []

for alpha in lasso_alphas:
    # Initialise and train the model
    lasso = Lasso(alpha=alpha, max_iter=10000) 
    lasso.fit(X_train, y_train)
    
    # Make predictions
    train_preds = lasso.predict(X_train)
    test_preds = lasso.predict(X_test)
    
    # Calculate and store the MAE
    train_mae_scores_lasso.append(mean_absolute_error(y_train, train_preds))
    test_mae_scores_lasso.append(mean_absolute_error(y_test, test_preds))

print("Tested all Lasso alphas using our simple loop!")

In [ ]:
# Plot train and test scores against alpha
plt.figure(figsize=(8, 4))
plt.plot(lasso_alphas, train_mae_scores_lasso, label='Train Error (MAE)', marker='o', color='green')
plt.plot(lasso_alphas, test_mae_scores_lasso, label='Test Error (MAE)', marker='s', color='purple')

plt.xscale('log') 
plt.xlabel('Alpha (Penalty Level)')
plt.ylabel('Mean Absolute Error')
plt.title('Lasso Regression: Error vs Alpha')
plt.legend()
plt.show()


In [ ]:
# Best alpha value

# 1. Find the lowest test error
lowest_test_error_lasso = min(test_mae_scores_lasso)

# 2. Find exactly what position (index) that lowest error is at
best_index_lasso = test_mae_scores_lasso.index(lowest_test_error_lasso)

# 3. Grab the winning alpha from alphas list
best_alpha_lasso = lasso_alphas[best_index_lasso]

print(f"The best Lasso alpha value found is: {best_alpha_lasso}")


# Best score (negative MAE)

best_score_lasso = -lowest_test_error_lasso

print(f"The best Lasso score (Negative MAE) is: {best_score_lasso:.4f}")

#### **3.3.3** <font color =red> [4 marks] </font>
Fine tune by taking a closer range of alpha based on the previous result.

In [ ]:
# List of alphas to tune for Lasso regularization

multipliers = [0.5, 0.75, 0.9, 1.0, 1.1, 1.25, 1.5]
fine_tune_alphas_lasso = [best_alpha_lasso * m for m in multipliers]

fine_tune_scores_lasso = []

# 3. Run the exact same loop used earlier
for alpha in fine_tune_alphas_lasso:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train, y_train)
    test_preds = lasso.predict(X_test)
    fine_tune_scores_lasso.append(mean_absolute_error(y_test, test_preds))

final_best_mae_lasso = min(fine_tune_scores_lasso)
best_index_lasso = fine_tune_scores_lasso.index(final_best_mae_lasso)
final_best_alpha_lasso = fine_tune_alphas_lasso[best_index_lasso]

print("Lasso Fine-tuning complete!")
print(f"The exact best Lasso alpha is: {final_best_alpha_lasso:.6f}") 
print(f"The final best Lasso score (Negative MAE) is: {-final_best_mae_lasso:.4f}")

In [ ]:
# Tuning Lasso hyperparameters

# 1. Initialize the final Lasso model with fine-tuned best alpha
best_lasso_model = Lasso(alpha=final_best_alpha_lasso, max_iter=10000)

# 2. Fit the optimized model to training data
best_lasso_model.fit(X_train, y_train)

# 3. Extract and display the coefficients 
lasso_coefs = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': best_lasso_model.coef_
}).sort_values(by='Coefficient', ascending=False)

print("Lasso Model Coefficients (Sorted by Impact):")
print(lasso_coefs)
print("-" * 30)

# 4. Evaluate on the hidden test set
final_lasso_preds = best_lasso_model.predict(X_test)
final_lasso_rmse = np.sqrt(mean_squared_error(y_test, final_lasso_preds))
final_lasso_r2 = r2_score(y_test, final_lasso_preds)

print("Final Optimized Lasso Model Performance:")
print(f"RMSE: {final_lasso_rmse:.4f}")
print(f"R-squared: {final_lasso_r2:.4f}")

In [ ]:
# Plot train and test scores against alpha

fine_tune_train_scores = []
fine_tune_test_scores = []

for alpha in fine_tune_alphas_lasso:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train, y_train)
    
    train_preds = lasso.predict(X_train)
    test_preds = lasso.predict(X_test)
    
    fine_tune_train_scores.append(mean_absolute_error(y_train, train_preds))
    fine_tune_test_scores.append(mean_absolute_error(y_test, test_preds))

plt.figure(figsize=(8, 4))
plt.plot(fine_tune_alphas_lasso, fine_tune_train_scores, label='Train Error (MAE)', marker='o', color='green')
plt.plot(fine_tune_alphas_lasso, fine_tune_test_scores, label='Test Error (MAE)', marker='s', color='purple')

# Using a standard linear scale this time since fine-tuned alphas are very close together
plt.xlabel('Alpha (Fine-Tuned Penalty Level)')
plt.ylabel('Mean Absolute Error')
plt.title('Lasso Fine-Tuning: Error vs Alpha')
plt.legend()
plt.show()

In [ ]:
# Best alpha value

# 1. Find the lowest test error from fine-tuning step
lowest_fine_tune_error = min(fine_tune_test_scores)

# 2. Find exactly what position (index) that lowest error is at
best_index_fine = fine_tune_test_scores.index(lowest_fine_tune_error)

# 3. Use that position to grab the winning alpha from fine-tuned list
final_best_alpha = fine_tune_alphas_lasso[best_index_fine]

print(f"The best fine-tuned Lasso alpha value is: {final_best_alpha:.6f}")

# Best score (negative MAE)

final_best_score = -lowest_fine_tune_error

print(f"The best fine-tuned Lasso score (Negative MAE) is: {final_best_score:.4f}")

In [ ]:
# Set best alpha for Lasso regression
final_lasso_model = Lasso(alpha=final_best_alpha, max_iter=10000)

print(f"Final Lasso model initialized with optimal alpha: {final_best_alpha:.6f}")

# Fit the Lasso model on scaled training data
# Get the coefficients of the fitted model

final_lasso_model.fit(X_train, y_train)

# 2. Extract the coefficients into a clean DataFrame
lasso_coefs = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': final_lasso_model.coef_
})

print("Model fitted successfully! Here is a peek at the Lasso coefficients:")
print("(Note: You might see some coefficients shrunk to exactly 0.0!)")
print(lasso_coefs.head())

In [ ]:
# Check the coefficients for each feature

sorted_lasso_coefs = lasso_coefs.sort_values(by='Coefficient', ascending=False)

print("Final Lasso Model Coefficients:")
print("-" * 40)
print(sorted_lasso_coefs)
print("-" * 40)

# 2. Count exactly how many features Lasso came to 0.0
features_ignored = (sorted_lasso_coefs['Coefficient'] == 0).sum()
print(f"Number of features completely ignored by Lasso (Coefficient = 0.0): {features_ignored}")

In [ ]:
# Evaluate the Lasso model on the test data

final_lasso_preds = final_lasso_model.predict(X_test)

# 2. Calculate the final performance metrics to see 
final_lasso_rmse = np.sqrt(mean_squared_error(y_test, final_lasso_preds))
final_lasso_r2 = r2_score(y_test, final_lasso_preds)

print("Final Optimized Lasso Model Performance:")
print(f"RMSE: {final_lasso_rmse:.4f}")
print(f"R-squared: {final_lasso_r2:.4f}")

### **3.4 Regularisation Comparison & Analysis** <font color =red> [5 marks] </font>

#### **3.4.1** <font color =red> [2 marks] </font>
Compare the evaluation metrics for each model.

In [ ]:
# Compare metrics for each model

comparison_data = {
    'Model': ['Ridge Regression', 'Lasso Regression'],
    'Optimal Alpha': [best_ridge_model.alpha, final_lasso_model.alpha],
    'RMSE': [final_rmse, final_lasso_rmse],
    'R-squared': [final_r2, final_lasso_r2]
}

# Convert the dictionary into a pandas DataFrame 
comparison_df = pd.DataFrame(comparison_data)

print("Final Model Comparison:")
print("-" * 65)
print(comparison_df.to_string(index=False)) 
print("-" * 65)

if final_lasso_rmse < final_rmse:
    print("\nWinner: Lasso Regression achieved the lower Error (RMSE)!")
else:
    print("\nWinner: Ridge Regression achieved the lower Error (RMSE)!")

#### **3.4.2** <font color =red> [3 marks] </font>
Compare the coefficients for the three models.

Also visualise a few of the largest coefficients and the coefficients of features dropped by Lasso.

In [ ]:
# Compare highest coefficients and coefficients of eliminated features
lin_model = LinearRegression().fit(X_train, y_train)
coefs = pd.DataFrame({
    'Feature': X_train.columns, 'Linear': lin_model.coef_, 
    'Ridge': best_ridge_model.coef_, 'Lasso': final_lasso_model.coef_
})

# 2. Select the top 5 most best features and 3 features Lasso dropped
top_5 = coefs.reindex(coefs['Ridge'].abs().sort_values(ascending=False).index).head(5)
dropped = coefs[coefs['Lasso'] == 0].head(3)
plot_data = pd.concat([top_5, dropped]).drop_duplicates().set_index('Feature')

# 3. Plot the comparison
plot_data.plot(kind='bar', figsize=(10, 5), colormap='viridis')
plt.title('Coefficient Comparison: Top Features vs Lasso-Dropped')
plt.axhline(0, color='black', linewidth=1)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## **4 Conclusion & Key Takeaways**  <font color =red> [10 marks] </font>

What did you notice by performing regularisation? Did the model performance improve? If not, then why? Did you find overfitting or not? Was the data sufficent? Is a linear model sufficient?

#### **4.1 Conclude with outcomes and insights gained** <font color =red> [10 marks] </font>

In [ ]:
# Answer is in the report pdf.